Region Lookup Table - Stats Can SCCAI
=====================================

SCCAI = "Standard Classification of Countries and Areas of Interest"

In [ ]:
import numpy as np
import pandas as pd

from utils.etl_utils import merge_columns_safe
from utils.paths import STG_STATCAN_DIR

In [2]:
# Main SCCAI 2019 country registry
URL_SCCAI_2019 = r'https://www.statcan.gc.ca/en/statistical-programs/document/sccai-2019-structure.csv'

# SCCAI 2019 social-statistics variant
URL_SOCIAL_2019 = r'https://www.statcan.gc.ca/en/statistical-programs/document/sccai-2019-var-structure.csv'

In [3]:
sccai_df = pd.read_csv(URL_SCCAI_2019)  # Read CSV directly from URL
sccai_social_df = pd.read_csv(URL_SOCIAL_2019)

original_sccai_df  = sccai_df.copy(deep=True)
original_social_df = sccai_social_df.copy(deep=True)

In [4]:
sccai_df = sccai_df.replace(' ', np.nan)
sccai_df.head(5)

,Code,Countries and Areas of Interest,Num-3,Alpha-2,Alpha-3,Footnotes
0,11124,Canada,124,CA,CAN,NaN
1,11304,Greenland,304,GL,GRL,NaN
2,11666,Saint Pierre and Miquelon,666,PM,SPM,NaN
3,11840,United States of America,840,US,USA,NaN
4,12084,Belize,84,BZ,BLZ,NaN


In [5]:
sccai_df = (
    sccai_df
    .rename(columns={
        'Code': 'sccai_code',
        'Countries and Areas of Interest': 'regions',
        'Num-3': 'm49_code',
        'Alpha-2': 'iso_alpha_2_code',
        'Alpha-3': 'iso_alpha_3_code',
        'Footnotes': 'sccai_notes',
    })
)

In [6]:
sccai_social_df = sccai_social_df.replace(' ', np.nan)
sccai_social_df.head(5)

,Level,Hierarchical structure,Code,Parent Code,Class title,Footnotes
0,1,Geographical macro-regions,1,NaN,Americas,NaN
1,2,Geographical sub-regions,11,1.0,North America,"North America includes Canada, Greenland, Sain..."
2,3,Countries and areas of interest,11124,11.0,Canada,NaN
3,3,Countries and areas of interest,11304,11.0,Greenland,NaN
4,3,Countries and areas of interest,11666,11.0,Saint Pierre and Miquelon,NaN


In [7]:
sccai_social_df = (
    sccai_social_df
    .rename(columns={
        'Level': 'hierarchical_level',
        'Hierarchical structure': 'hierarchical_desc',
        'Code': 'sccai_code',
        'Parent Code': 'parent_sccai_code',
        'Class title': 'region',
        'Footnotes': 'sccai_social_notes',
    })
)

In [ ]:
# sccai_df.to_csv(STG_STATCAN_DIR / 'stg_statcan__sccai_regions.csv')
# sccai_social_df.to_csv(STG_STATCAN_DIR / 'stg_statcan__sccai_social_regions.csv')

In [8]:
sccai_df.iloc[47]


sccai_code                                14238
regions             Falkland Islands (Malvinas)
m49_code                                    238
iso_alpha_2_code                             FK
iso_alpha_3_code                            FLK
sccai_notes                                 NaN
Name: 47, dtype: object

In [10]:
sccai_combined_df = (
    sccai_social_df
    .merge(
        sccai_df[['sccai_code', 'm49_code', 'iso_alpha_2_code', 'iso_alpha_3_code', 'sccai_notes']],
        on='sccai_code',
        how='left'
    )
)

merged_footnotes = merge_columns_safe(sccai_combined_df,
                                      col1='sccai_social_notes',
                                      col2='sccai_notes')

sccai_combined_df['notes'] = merged_footnotes
sccai_combined_df = (
    sccai_combined_df
    .drop(columns={'sccai_social_notes', 'sccai_notes'})
    .convert_dtypes()
)

sccai_combined_df.insert(0, 'sccai_code_id', sccai_combined_df.index+1)
sccai_combined_df

,sccai_code_id,hierarchical_level,hierarchical_desc,sccai_code,parent_sccai_code,region,m49_code,iso_alpha_2_code,iso_alpha_3_code,notes
0,1,1,Geographical macro-regions,1,<NA>,Americas,<NA>,<NA>,<NA>,<NA>
1,2,2,Geographical sub-regions,11,1,North America,<NA>,<NA>,<NA>,"North America includes Canada, Greenland, Sain..."
2,3,3,Countries and areas of interest,11124,11,Canada,124,CA,CAN,<NA>
3,4,3,Countries and areas of interest,11304,11,Greenland,304,GL,GRL,<NA>
4,5,3,Countries and areas of interest,11666,11,Saint Pierre and Miquelon,666,PM,SPM,<NA>
...,...,...,...,...,...,...,...,...,...,...
271,272,2,Geographical sub-regions,61,6,Antarctica and Adjacent Islands,<NA>,<NA>,<NA>,<NA>
272,273,3,Countries and areas of interest,61010,61,Antarctica,10,AQ,ATA,<NA>
273,274,3,Countries and areas of interest,61074,61,Bouvet Island,74,BV,BVT,<NA>
274,275,3,Countries and areas of interest,61260,61,French Southern Territories,260,TF,ATF,<NA>


In [ ]:
sccai_filename = 'stg_statcan__sccai_regions.csv'
sccai_filepath = STG_STATCAN_DIR / sccai_filename
sccai_combined_df.to_csv(sccai_filepath, index=False)